<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Method: Logistic Regression

I will use Logistic Regression because my target variable, `is_declining_label`, is binary: 1 means the article is observed as declining and 0 means it is not.

Logistic Regression is a suitable first modeling method because it is simple, interpretable, and provides predicted probabilities. I can use the predicted probability of decline as a ranking score and compare the resulting ranking with my Week-4 rule-based baseline.

I chose this method instead of starting with a more complex model because the goal is not to reward complexity. The Week-5 model should provide an honest improvement over the Week-4 baseline using the same data, split, and Precision@K metric.

In [5]:
# Section 1: Method choice and why

method_name = "Logistic Regression"

method_reason = """
Logistic Regression was selected because the target variable is binary
(is_declining_label = 0 or 1). It provides a simple and interpretable
classification model that is appropriate for establishing a strong,
non-complex modeling baseline.

The dataset contains both numeric and categorical features, which can be
handled using the preprocessing pipeline already defined in this notebook.
Logistic Regression also provides predicted probabilities, allowing us to
evaluate ranking-based metrics such as Precision@20, Precision@50,
Precision@100, ROC-AUC, and Average Precision.

The goal is not to maximize complexity, but to determine whether a
well-validated model can improve upon the Week-4 baseline on the same
evaluation data.
"""

print("Selected method:", method_name)
print(method_reason)

Selected method: Logistic Regression

Logistic Regression was selected because the target variable is binary
(is_declining_label = 0 or 1). It provides a simple and interpretable
classification model that is appropriate for establishing a strong,
non-complex modeling baseline.

The dataset contains both numeric and categorical features, which can be
handled using the preprocessing pipeline already defined in this notebook.
Logistic Regression also provides predicted probabilities, allowing us to
evaluate ranking-based metrics such as Precision@20, Precision@50,
Precision@100, ROC-AUC, and Average Precision.

The goal is not to maximize complexity, but to determine whether a
well-validated model can improve upon the Week-4 baseline on the same
evaluation data.



### Why Logistic Regression?

I selected Logistic Regression because the target variable `is_declining_label` is binary. It is an appropriate first modeling method because it is relatively simple, interpretable, and produces probability estimates that are useful for ranking content.

The model is also suitable for the available numeric and categorical features after preprocessing. I chose this method instead of immediately using a more complex model because the objective is to establish whether a reasonable supervised model can improve upon the Week-4 baseline without rewarding complexity alone.

The model will be evaluated against the Week-4 baseline using the same test data and the same metrics.

### 2. Split design: Grouped train/test split by client

I will use a grouped train/test split based on `client_id`.

The dataset contains multiple content items from the same client, so randomly splitting individual rows could place content from the same client in both training and test data. That could make the evaluation look more optimistic than it should be.

I will therefore keep each client entirely in either the training or test set. I will use an 80/20 split with a fixed random seed so the result is reproducible.

The target is `is_declining_label`. The identifiers `content_id` and `client_id` will not be used as model features. `client_id` is used only for grouping the split.

The same test set and the same Precision@K metric will be used for both the Week-4 rule-based baseline and the Logistic Regression model.

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit

# Load the dataset directly from the GitHub repository
DATA_URL = "https://raw.githubusercontent.com/nomanamir20/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)

# Keep the modeling lane used in earlier weeks
df = df[df["content_type"] == "keyword article"].copy()

print("Lane shape:", df.shape)

# Create the binary observed target
# 1 = declining, 0 = not declining
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())
print("\nTarget rate:")
print(df["is_declining_label"].mean())

# Grouped 80/20 split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("\nTraining rows:", len(train))
print("Test rows:", len(test))

print("Training clients:", train["client_id"].nunique())
print("Test clients:", test["client_id"].nunique())

# Verify that no client appears in both sets
overlap = set(train["client_id"]).intersection(
    set(test["client_id"])
)

print("Client overlap:", len(overlap))

print("\nTraining target rate:")
print(train["is_declining_label"].mean())

print("\nTest target rate:")
print(test["is_declining_label"].mean())

Dataset shape: (30000, 44)
Lane shape: (27207, 44)

Target distribution:
is_declining_label
1    15262
0    11945
Name: count, dtype: int64

Target rate:
0.5609585768368435

Training rows: 21425
Test rows: 5782
Training clients: 24
Test clients: 7
Client overlap: 0

Training target rate:
0.576242707117853

Test target rate:
0.5043237634036666


### 3. Logistic Regression vs Week-4 Rule-Based Baseline

I will train a Logistic Regression model using the training portion of the grouped client split.

The model will produce a probability that each page is declining. I will use this probability as the model's ranking score.

For a fair comparison, I will evaluate both the Logistic Regression ranking and the Week-4 rule-based baseline on the same held-out test set.

The primary ranking metric is Precision@20 because the original task is to prioritize a small review queue. I will also report ROC-AUC and Average Precision as supporting classification metrics.

The model will not use `trend_direction`, `trend_pct`, `content_id`, or `client_id` as features because they would either leak the target or act as identifiers rather than meaningful predictive signals.

In [7]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score


# ---------------------------------------------------------
# 1. Define model features
# ---------------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

feature_columns = numeric_features + categorical_features


# ---------------------------------------------------------
# 2. Keep only features that actually exist
# ---------------------------------------------------------

numeric_features = [
    col for col in numeric_features
    if col in train.columns
]

categorical_features = [
    col for col in categorical_features
    if col in train.columns
]

feature_columns = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(feature_columns))


# ---------------------------------------------------------
# 3. Prepare X and y
# ---------------------------------------------------------

X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()

y_train = train["is_declining_label"]
y_test = test["is_declining_label"]


# ---------------------------------------------------------
# 4. Preprocessing
# ---------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True
        ))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)


# ---------------------------------------------------------
# 5. Logistic Regression
# ---------------------------------------------------------

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


# ---------------------------------------------------------
# 6. Train model
# ---------------------------------------------------------

model.fit(X_train, y_train)

print("\nLogistic Regression training completed.")


# ---------------------------------------------------------
# 7. Generate model ranking scores
# ---------------------------------------------------------

model_scores = model.predict_proba(X_test)[:, 1]

print("Model scores generated.")


# ---------------------------------------------------------
# 8. Recreate Week-4 baseline on the SAME test set
# ---------------------------------------------------------

baseline_test = test[
    [
        "content_id",
        "search_volume",
        "ctr",
        "days_since_last_update",
        "is_declining_label"
    ]
].copy()

baseline_test["search_volume"] = (
    baseline_test["search_volume"].fillna(0)
)

baseline_test["ctr"] = (
    baseline_test["ctr"].fillna(0)
)

baseline_test["days_since_last_update"] = (
    baseline_test["days_since_last_update"].fillna(0)
)


# Same Week-4 weighting:
# 40% search volume
# 30% low CTR
# 30% stale content

max_search_volume = baseline_test["search_volume"].max()

if max_search_volume > 0:
    sv_score = (
        baseline_test["search_volume"] /
        max_search_volume
    )
else:
    sv_score = 0

max_age = baseline_test["days_since_last_update"].max()

if max_age > 0:
    freshness_score = (
        baseline_test["days_since_last_update"] /
        max_age
    )
else:
    freshness_score = 0

ctr_score = 1 - baseline_test["ctr"]

baseline_test["baseline_score"] = (
    0.4 * sv_score +
    0.3 * ctr_score +
    0.3 * freshness_score
)


# ---------------------------------------------------------
# 9. Precision@K helper
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k):
    ranking = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    ranking = ranking.sort_values(
        "score",
        ascending=False
    )

    top_k = ranking.head(min(k, len(ranking)))

    return top_k["y_true"].mean()


# ---------------------------------------------------------
# 10. Evaluate both rankings
# ---------------------------------------------------------

k_values = [20, 50, 100]

results = []

for k in k_values:

    model_p = precision_at_k(
        y_test,
        model_scores,
        k
    )

    baseline_p = precision_at_k(
        baseline_test["is_declining_label"],
        baseline_test["baseline_score"],
        k
    )

    results.append({
        "Metric": f"Precision@{k}",
        "Week-4 Baseline": baseline_p,
        "Logistic Regression": model_p
    })


# Supporting classification metrics

model_auc = roc_auc_score(
    y_test,
    model_scores
)

model_ap = average_precision_score(
    y_test,
    model_scores
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_test["baseline_score"]
)

baseline_ap = average_precision_score(
    y_test,
    baseline_test["baseline_score"]
)

results.append({
    "Metric": "ROC-AUC",
    "Week-4 Baseline": baseline_auc,
    "Logistic Regression": model_auc
})

results.append({
    "Metric": "Average Precision",
    "Week-4 Baseline": baseline_ap,
    "Logistic Regression": model_ap
})


# ---------------------------------------------------------
# 11. Final comparison table
# ---------------------------------------------------------

comparison_table = pd.DataFrame(results)

comparison_table["Difference"] = (
    comparison_table["Logistic Regression"] -
    comparison_table["Week-4 Baseline"]
)

print("\nMODEL VS BASELINE")
display(comparison_table.round(4))

Numeric features: 22
Categorical features: 8
Total model features: 30

Logistic Regression training completed.
Model scores generated.

MODEL VS BASELINE


,Metric,Week-4 Baseline,Logistic Regression,Difference
0,Precision@20,0.5000,0.8500,0.3500
1,Precision@50,0.4400,0.7400,0.3000
2,Precision@100,0.4600,0.7200,0.2600
3,ROC-AUC,0.4595,0.6161,0.1566
4,Average Precision,0.4793,0.6097,0.1304


### 4. Error analysis

The Logistic Regression model is substantially stronger than the Week-4 rule-based baseline on the same held-out test set. I will now inspect the highest-confidence mistakes to understand where the model still fails.

I will focus on false positives and false negatives rather than only reporting aggregate metrics. This helps determine whether the model is making systematic errors or struggling with ambiguous cases.

In [8]:
# ---------------------------------------------------------
# Error analysis
# ---------------------------------------------------------

error_analysis = test[
    [
        "content_id",
        "client_id",
        "content_type",
        "trend_direction",
        "trend_pct",
        "search_volume",
        "ctr",
        "days_since_last_update"
    ]
].copy()

error_analysis["actual"] = y_test.values
error_analysis["predicted_probability"] = model_scores

error_analysis["predicted"] = (
    error_analysis["predicted_probability"] >= 0.5
).astype(int)

error_analysis["error_type"] = np.select(
    [
        (error_analysis["actual"] == 0) &
        (error_analysis["predicted"] == 1),

        (error_analysis["actual"] == 1) &
        (error_analysis["predicted"] == 0)
    ],
    [
        "False Positive",
        "False Negative"
    ],
    default="Correct"
)

print("Error counts:")
print(error_analysis["error_type"].value_counts())

print("\nError rates:")
print(
    error_analysis["error_type"]
    .value_counts(normalize=True)
    .round(4)
)

print("\nHighest-confidence false positives:")

false_positives = (
    error_analysis[
        error_analysis["error_type"] == "False Positive"
    ]
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .head(10)
)

display(false_positives)

print("\nHighest-confidence false negatives:")

false_negatives = (
    error_analysis[
        error_analysis["error_type"] == "False Negative"
    ]
    .sort_values(
        "predicted_probability",
        ascending=True
    )
    .head(10)
)

display(false_negatives)

Error counts:
error_type
Correct           3394
False Negative    1584
False Positive     804
Name: count, dtype: int64

Error rates:
error_type
Correct           0.5870
False Negative    0.2740
False Positive    0.1391
Name: proportion, dtype: float64

Highest-confidence false positives:


,content_id,client_id,content_type,trend_direction,trend_pct,search_volume,ctr,days_since_last_update,actual,predicted_probability,predicted,error_type
10175,content_374e795aab68,client_f369cb89fc,keyword article,stable,0.0,880.0,0.85,20,0,0.854858,1,False Positive
26614,content_7be5f150dc65,client_f369cb89fc,keyword article,up,213.9,10.0,0.00,20,0,0.833143,1,False Positive
8016,content_c94a53e3bfb8,client_f369cb89fc,keyword article,up,71.0,0.0,0.23,20,0,0.806605,1,False Positive
18531,content_d10f9ce1e0cd,client_4e07408562,keyword article,stable,2.2,30.0,0.60,104,0,0.794239,1,False Positive
17602,content_3b26815717ae,client_f369cb89fc,keyword article,up,143.6,10.0,0.09,20,0,0.793350,1,False Positive
13631,content_d274ac4158ef,client_4e07408562,keyword article,stable,-11.8,40.0,0.01,26,0,0.792141,1,False Positive
8139,content_7fa63804b8f1,client_4e07408562,keyword article,stable,-19.1,10.0,0.04,104,0,0.792072,1,False Positive
6903,content_c84a0ab98e90,client_f369cb89fc,keyword article,stable,17.2,0.0,0.03,20,0,0.790901,1,False Positive
11089,content_a058e966b9a6,client_f369cb89fc,keyword article,up,181.3,0.0,1.27,20,0,0.790263,1,False Positive
4905,content_f0d98be4b42c,client_4e07408562,keyword article,stable,6.2,10.0,0.15,104,0,0.788032,1,False Positive



Highest-confidence false negatives:


,content_id,client_id,content_type,trend_direction,trend_pct,search_volume,ctr,days_since_last_update,actual,predicted_probability,predicted,error_type
17127,content_8818fd6d967f,client_4e07408562,keyword article,down,-21.5,10.0,1.06,104,1,0.053887,0,False Negative
28214,content_13bbd72aea33,client_e29c9c180c,keyword article,down,-100.0,40.0,0.00,104,1,0.079410,0,False Negative
21565,content_9532f197bbc8,client_4e07408562,keyword article,down,-37.3,10.0,0.87,104,1,0.079774,0,False Negative
8407,content_d1e915d03c28,client_4e07408562,keyword article,down,-100.0,140.0,0.00,104,1,0.096697,0,False Negative
5059,content_a638a00cdb70,client_8722616204,keyword article,down,-100.0,0.0,0.00,8,1,0.098403,0,False Negative
26413,content_bdd7c88a58ed,client_4e07408562,keyword article,down,-73.3,10.0,0.84,14,1,0.100650,0,False Negative
21819,content_4c36c775b818,client_4e07408562,keyword article,down,-33.2,40.0,0.41,20,1,0.107321,0,False Negative
15404,content_90d7c5385cda,client_4ec9599fc2,keyword article,down,-98.1,0.0,0.00,20,1,0.118948,0,False Negative
18853,content_fb7fb643eff7,client_4e07408562,keyword article,down,-36.4,320.0,0.86,104,1,0.119933,0,False Negative
28858,content_6989c356365e,client_4e07408562,keyword article,down,-50.9,320.0,0.25,35,1,0.120673,0,False Negative


## 5. Self-Check

- [x] The model is compared against the Week-4 baseline.
- [x] The same dataset and evaluation setup are used for the comparison.
- [x] The train/test split is grouped by client to reduce client-level leakage.
- [x] The target variable is checked for missing values and class distribution.
- [x] Logistic Regression was selected because it provides a strong, interpretable baseline for binary classification.
- [x] Useful evaluation metrics are reported, including Precision@20, Precision@50, Precision@100, ROC-AUC, and Average Precision.
- [x] Model performance is compared directly with the Week-4 baseline.
- [x] False positives and false negatives are examined to understand model errors.
- [x] High-confidence incorrect predictions are inspected for possible patterns.
- [x] The model improves over the Week-4 baseline on all reported metrics.
- [x] The results are interpreted rather than judging the model only by complexity.

### Final conclusion

Logistic Regression outperformed the Week-4 baseline across all evaluated metrics. Precision@20 improved from 0.5000 to 0.8500, Precision@50 from 0.4400 to 0.7400, and Precision@100 from 0.4600 to 0.7200. ROC-AUC improved from 0.4595 to 0.6161, while Average Precision improved from 0.4793 to 0.6097.

The error analysis shows both false positives and false negatives. Inspecting the highest-confidence errors helps identify cases where the available features do not fully explain the target outcome. Overall, the model provides a meaningful improvement over the Week-4 baseline while remaining relatively simple and interpretable.